# FMP Equities

Read-first examples for FMP-backed equity data in Quant Warehouse. Set `RUN_REFRESH = True` only when you want to download missing data.

In [ ]:
from __future__ import annotations

from dataclasses import asdict

import pandas as pd
from openbb import obb

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

EQUITY_SYMBOL = "AAPL"

In [ ]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

## Available Equity Sections

In [ ]:
equity_sections = ["prices", "profile", *FMP_HISTORICAL_EQUITY_SECTIONS, *FMP_EXTENDED_EQUITY_SECTIONS]
equity_route_libraries = {
    "prices": EQUITY_PRICES_LIBRARY,
    "profile": "profile",
    **{section: fundamental_library(section) for section in FMP_HISTORICAL_EQUITY_SECTIONS},
    **{section: fundamental_library(section) for section in FMP_EXTENDED_EQUITY_SECTIONS},
}
route_table(equity_route_libraries)

## Local Storage Coverage

In [ ]:
state_table(EQUITY_SYMBOL, equity_sections)

## Prices and Profile

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_prices(EQUITY_SYMBOL, providers=[PROVIDER])
    warehouse.refresh_profile(EQUITY_SYMBOL, provider=PROVIDER)

preview(warehouse.read_prices(EQUITY_SYMBOL, provider=PROVIDER))

In [ ]:
profile = warehouse.read_profile(EQUITY_SYMBOL, provider=PROVIDER)
asdict(profile) if profile is not None else None

## Core Fundamentals

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_fundamentals(
        EQUITY_SYMBOL,
        sections=["income", "balance", "cash", "metrics", "ratios"],
        providers=[PROVIDER],
        period="quarter",
    )

{
    "income": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="income", provider=PROVIDER)),
    "balance": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="balance", provider=PROVIDER)),
    "cash": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="cash", provider=PROVIDER)),
    "ratios": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="ratios", provider=PROVIDER)),
}

## Events, Estimates, Ownership, and Calendars

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_fundamentals(
        EQUITY_SYMBOL,
        sections=["dividends", "historical_splits", "filings", "estimates_price_target", "ownership_insider_trading"],
        providers=[PROVIDER],
    )

{
    "dividends": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="dividends", provider=PROVIDER)),
    "historical_splits": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="historical_splits", provider=PROVIDER)),
    "filings": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="filings", provider=PROVIDER)),
    "estimates_price_target": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="estimates_price_target", provider=PROVIDER)),
}

In [ ]:
calendar_routes = {section: fundamental_library(section) for section in EQUITY_CALENDAR_SECTIONS}
route_table(calendar_routes)

In [ ]:
if RUN_REFRESH:
    warehouse.equity_calendar.refresh_all(provider=PROVIDER, start_date="2024-01-01")

state_table("EQUITY_CALENDAR", EQUITY_CALENDAR_SECTIONS)